# Native electric-grid optimization
Keep three sources, four consumers, twelve decisions and six bounded-slack bits. Run all four QAOA layers, original100iteration mean-cost search, actual optimized and zero-angle uniform samples, all network/trace/histogram/table stages and a fresh exact constrained classical reference. Public native Nelder-Mead and exact local means replace proprietary optimization and shot estimates; resynthesize each vertex. Preserve raw supply cost, joint penalized cost and decision feasibility separately.

In [ ]:
import json
import random
from pathlib import Path

import torch
from IPython.display import Image, display

from examples.synthesis.applications.optimization.electric_grid_optimization.overview.model import (
    COSTS,
    assignment,
)
from examples.synthesis.applications.optimization.electric_grid_optimization.overview.plots import (
    plot_network,
    plot_results,
)
from examples.synthesis.applications.optimization.electric_grid_optimization.overview.quantum import (
    build_program,
)
from examples.synthesis.applications.optimization.electric_grid_optimization.overview.workflow import (
    INITIAL_PARAMETERS,
    run_workflow,
)
from flagquantum.drawer import draw

random.seed(8)
torch.manual_seed(8)


In [ ]:
sources = ("A1", "A2", "A3")
consumers = ("B1", "B2", "B3", "B4")
print("Original cost matrix:", COSTS)
artifacts = Path("artifacts/development/electric-grid")
initial_graph = plot_network(artifacts / "supply-network.png")
display(Image(filename=str(initial_graph)))


In [ ]:
print("Twelve binary source/consumer decisions.")
print(
    "Each source supplies at most two consumers; each consumer has exactly one source."
)
print("Three two-bit slack registers; both bits have weight one, encoding0,1,1,2.")
print("Objective: raw supply cost plus ten times squared equality residuals.")


In [ ]:
decisions = tuple(assignment(word) for word in range(4096))
print("Decision assignments:", len(decisions))
print("Feasible decision assignments:", sum(value.feasible for value in decisions))
print("No classical solution selected before quantum optimization.")


In [ ]:
initial_program = build_program(INITIAL_PARAMETERS)
print("Four source layers,18totalbits,zero temporary workspace.")
print("Native default synthesis objective:", initial_program.selection.objective)
print("Fresh synthesis per parameter tuple; no runtime-parameter binding claim.")


In [ ]:
print(initial_program.report["resources"])
print(draw(initial_program.circuit, show_all_qubits=True))


In [ ]:
workflow = run_workflow(maxiter=100, shots=1000, seed=2718)
print("Actual objective evaluations:", len(workflow.trace))
print("Optimizer iterations:", workflow.optimizer.iterations)


In [ ]:
plot_paths = plot_results(workflow, artifacts)
display(Image(filename=str(plot_paths[1])))


In [ ]:
print("Top five actual joint observations, with slack and penalized cost:")
for row in workflow.samples[:5]:
    print(row, "penalized cost=", row.penalized_cost)
assert sum(row.count for row in workflow.samples) == 1000


In [ ]:
print("Actual zero-angle native uniform samples:", len(workflow.uniform))
assert sum(row.count for row in workflow.uniform) == 1000


In [ ]:
display(Image(filename=str(plot_paths[2])))


In [ ]:
best = workflow.best_observed
print("Source reported penalized cost:", best.penalized_cost)
print("Raw supply cost:", best.assignment.raw_cost)
print("Decision feasibility:", best.assignment.feasible)
print("Residual penalty:", best.penalty)
print("Source/consumer decision table:")
for source_name, values in zip(sources, best.assignment.rows, strict=True):
    print(source_name, values)
display(Image(filename=str(plot_paths[3])))


In [ ]:
print("Separate exact constrained solution, calculated after sampling:")
print(
    "Fresh exhaustive decision solver replaces external Couenne; no trajectory claim."
)
for reference in workflow.reference:
    print(reference)
display(Image(filename=str(plot_paths[4])))
report = {
    "decision_bits": 12,
    "slack_bits": 6,
    "width": 18,
    "layers": 4,
    "maxiter": 100,
    "iterations": workflow.optimizer.iterations,
    "evaluations": len(workflow.trace),
    "quantile": 1,
    "optimized_shots": sum(row.count for row in workflow.samples),
    "uniform_shots": sum(row.count for row in workflow.uniform),
    "best_joint_word": best.joint_word,
    "best_cost": best.penalized_cost,
    "best_raw_cost": best.assignment.raw_cost,
    "best_penalty": best.penalty,
    "best_feasible": best.assignment.feasible,
    "reference_cost": workflow.reference[0].raw_cost,
    "reference_words": [value.word for value in workflow.reference],
    "trace": [value.cost for value in workflow.trace],
}
print("ARTIFACT_JSON=" + json.dumps(report))
